# Valutazioni finali di stage – analisi per il management

Notebook riproducibile dall'inizio alla fine. Ogni step è separato da un'intestazione;
i trattamenti sui dati vengono applicati **solo dopo conferma esplicita** e sono documentati
nel *Registro assunzioni e limiti* in fondo a ogni step.

**Input:** `data/INT_VAL_ASSESSMENT_SET.csv` (non versionato nel repo)
**Output:** tabelle in `outputs/tables/`, grafici in `outputs/figures/`

## Step 0 – Struttura del file (nessuna analisi, nessun trattamento)

**Approccio.** Il file viene letto con *tutte le colonne come testo* (`dtype=str`, `keep_default_na=False`).

*Alternative considerate:*
- lettura con i default di pandas: scartata, perché converte in silenzio `"N/A"` e le celle vuote in `NaN`
  e assegna tipi diversi a colonne omogenee (es. `Listening` diventa testo a causa di un solo valore `BBBB`),
  nascondendo proprio le anomalie che vogliamo vedere;
- lettura dall'`.xlsx`: usato solo come controllo incrociato (deve essere identico al CSV).

Così ogni valore viene visto esattamente come scritto nel file; tipi e anomalie sono *diagnosticati*, non corretti.

In [1]:
# --- Import e percorsi -------------------------------------------------------
from pathlib import Path
from collections import Counter
import numpy as np
import pandas as pd

pd.set_option("display.width", 200)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 30)

DATA_CSV  = Path("data/INT_VAL_ASSESSMENT_SET.csv")
DATA_XLSX = Path("data/INT_VAL_ASSESSMENT_SET.xlsx")   # opzionale, solo per controllo incrociato
OUT_TAB   = Path("outputs/tables"); OUT_TAB.mkdir(parents=True, exist_ok=True)

# Colonne attese secondo la traccia (da verificare, non da dare per scontate)
SKILLS = ["Digital", "Teamwork", "Entrepreneurial Mindset", "Critical Thinking", "Creativity",
          "Planning", "Flexibility", "Cultural Awareness", "Writing", "Listening",
          "Time Management", "Responsibility", "Motivation", "Feedback Management",
          "Interpersonal Skills", "Leadership"]
SCORES = SKILLS + ["OVERALL"]                      # 17 colonne su scala attesa 1-10
CATEG  = ["NAZIONE", "SETTORE AZIENDALE", "AREA FUNZIONALE", "CORSO", "Hiring"]
EXPECTED_COLS = ["ID", "DURATA", "NAZIONE", "SETTORE AZIENDALE", "AREA FUNZIONALE", "CORSO"] + SCORES + ["Hiring"]

### 0.1 Controlli sul file grezzo (byte e righe)
Prima di usare pandas guardiamo il file "a mano": codifica, separatore, fine riga, numero di campi per riga.

In [2]:
raw = DATA_CSV.read_bytes()

# Codifica: se non è UTF-8 valido, cerchiamo i byte non ASCII per capire quale codifica usare
try:
    raw.decode("utf-8"); ENCODING = "utf-8"
except UnicodeDecodeError:
    ENCODING = "cp1252"   # nessun byte 0x80-0x9F -> cp1252 e latin-1 coincidono su questo file
non_ascii = Counter(b for b in raw if b >= 0x80)
print("Codifica usata:", ENCODING, "| byte non ASCII:", {hex(k): v for k, v in non_ascii.items()})
print("Esempio decodificato:", bytes([0xE0]).decode(ENCODING))

text  = raw.decode(ENCODING)
lines = [l for l in text.split("\r\n") if l != ""]
print("Fine riga CRLF:", "\r\n" in text)
print("Righe non vuote (incluso header):", len(lines))
print("Numero di ';' per riga:", Counter(l.count(";") for l in lines))

# Controlli automatici sulla struttura grezza
assert all(l.count(";") == 23 for l in lines), "righe con numero di campi diverso da 24"
N_ROWS_RAW = len(lines) - 1          # righe dati, header escluso

Codifica usata: cp1252 | byte non ASCII: {'0xe0': 634}
Esempio decodificato: à
Fine riga CRLF: True
Righe non vuote (incluso header): 4383
Numero di ';' per riga: Counter({23: 4383})


### 0.2 Caricamento (tutto come testo) e verifica delle colonne

In [3]:
df_raw = pd.read_csv(DATA_CSV, sep=";", encoding=ENCODING, dtype=str, keep_default_na=False)
print("Shape:", df_raw.shape)

# Verifica indipendente: righe lette da pandas == righe contate sul file grezzo
assert len(df_raw) == N_ROWS_RAW, (len(df_raw), N_ROWS_RAW)
# Verifica colonne attese: stessi nomi, stesso ordine, nessuno spazio spurio
assert list(df_raw.columns) == EXPECTED_COLS, set(df_raw.columns) ^ set(EXPECTED_COLS)
assert all(c == c.strip() for c in df_raw.columns)
print("Colonne verificate:", len(df_raw.columns))

Shape: (4382, 24)
Colonne verificate: 24


### 0.3 Profilo delle colonne
Per ogni colonna: valori distinti, celle vuote, `N/A` letterali, quanti valori sono numeri interi,
token non numerici, min/max dei valori numerici e (per i punteggi) quanti cadono fuori dalla scala 1-10.
Il "tipo diagnosticato" è quello che la colonna *dovrebbe* avere; il "tipo pandas default" mostra cosa
succederebbe con una lettura ingenua.

In [4]:
df_default = pd.read_csv(DATA_CSV, sep=";", encoding=ENCODING)   # solo per confronto dei tipi

rows = []
for c in df_raw.columns:
    s   = df_raw[c]
    num = pd.to_numeric(s, errors="coerce")                         # NaN dove non numerico
    tok = s[num.isna() & (s != "") & (s != "N/A")]                  # token anomali (né vuoto né N/A)
    is_num_col = num.notna().mean() > 0.9
    rows.append({
        "colonna": c,
        "tipo_pandas_default": str(df_default[c].dtype),
        "tipo_diagnosticato": (("intero, atteso 1-10" if c in SCORES else "intero") if is_num_col else "categorica (testo)"),
        "n_righe": len(s),
        "n_distinti": s.nunique(),
        "n_vuoti": int((s == "").sum()),
        "n_NA_letterale": int((s == "N/A").sum()),
        "n_numerici": int(num.notna().sum()),
        "token_anomali": {k: int(v) for k, v in tok.value_counts().items()} if is_num_col and len(tok) else "",
        "min": num.min() if is_num_col else "",
        "max": num.max() if is_num_col else "",
        "n_fuori_1_10": int(((num < 1) | (num > 10)).sum()) if c in SCORES else "",
        "n_spazi_iniziali_finali": int((s != s.str.strip()).sum()),
    })
profilo = pd.DataFrame(rows)
profilo.to_csv(OUT_TAB / "step0_profilo_colonne.csv", index=False)
profilo

,colonna,tipo_pandas_default,tipo_diagnosticato,n_righe,n_distinti,n_vuoti,n_NA_letterale,n_numerici,token_anomali,min,max,n_fuori_1_10,n_spazi_iniziali_finali
0,ID,int64,intero,4382,4382,0,0,4382,,1,4388,,0
1,DURATA,int64,intero,4382,247,0,0,4382,,20,45534,,0
2,NAZIONE,str,categorica (testo),4382,65,3,0,0,,,,,0
3,SETTORE AZIENDALE,str,categorica (testo),4382,38,0,0,0,,,,,131
4,AREA FUNZIONALE,str,categorica (testo),4382,23,1,0,0,,,,,0
5,CORSO,str,categorica (testo),4382,38,1,0,0,,,,,0
6,Digital,float64,"intero, atteso 1-10",4382,12,1,65,4316,,1.0,10.0,0,0
7,Teamwork,float64,"intero, atteso 1-10",4382,12,1,90,4291,,1.0,10.0,0,0
8,Entrepreneurial Mindset,float64,"intero, atteso 1-10",4382,13,1,153,4228,,1.0,7000.0,1,0
9,Critical Thinking,float64,"intero, atteso 1-10",4382,12,1,127,4254,,1.0,10.0,0,0


### 0.4 Variabili categoriche: valori distinti e conteggi
I valori sono mostrati con `repr()` per rendere visibili celle vuote (`''`) e spazi finali.

In [5]:
vc_list = []
for c in CATEG:
    vc = df_raw[c].value_counts(dropna=False)
    vc_list.append(pd.DataFrame({"variabile": c, "valore": vc.index, "n": vc.values,
                                 "pct": (vc.values / len(df_raw) * 100).round(2)}))
    # Verifica: la somma dei conteggi deve dare il totale righe
    assert vc.sum() == len(df_raw)
valori_cat = pd.concat(vc_list, ignore_index=True)
valori_cat.to_csv(OUT_TAB / "step0_valori_categoriche.csv", index=False)

for c in CATEG:
    sub = valori_cat[valori_cat.variabile == c]
    print(f"\n=== {c}: {len(sub)} valori distinti (n totale = {sub.n.sum()}) ===")
    print(sub.assign(valore=sub.valore.map(repr))[["valore", "n", "pct"]].to_string(index=False))


=== NAZIONE: 65 valori distinti (n totale = 4382) ===
                            valore    n   pct
                          'ITALIA' 3403 77.66
                         'FRANCIA'  157  3.58
                        'GERMANIA'  114  2.60
                          'BELGIO'   87  1.99
'GRAN BRETAGNA E IRLANDA DEL NORD'   81  1.85
                     'LUSSEMBURGO'   63  1.44
                        'SVIZZERA'   63  1.44
           "STATI UNITI D'AMERICA"   48  1.10
        'CINA REPUBBLICA POPOLARE'   36  0.82
                          'SPAGNA'   33  0.75
                         'TURCHIA'   26  0.59
                     'PAESI BASSI'   25  0.57
           'REPUBBLICA DI POLONIA'   21  0.48
                         'ROMANIA'   18  0.41
                       'SINGAPORE'   17  0.39
             'EMIRATI ARABI UNITI'   16  0.37
                          'GRECIA'   14  0.32
                       'DANIMARCA'   10  0.23
                         'MAROCCO'   10  0.23
                      'PO

### 0.5 Hiring – valori presenti (con verifica indipendente)

In [6]:
hiring_vc = df_raw["Hiring"].value_counts()
print(hiring_vc.rename(index=repr))

# Verifica indipendente: filtro semplice vs value_counts
for v in hiring_vc.index:
    assert (df_raw["Hiring"] == v).sum() == hiring_vc[v]
print("\nRighe con Hiring diverso da SI/NO:")
print(df_raw.loc[~df_raw["Hiring"].isin(["SI", "NO"]), ["ID", "OVERALL", "Hiring"]])

Hiring
'SI'      2494
'NO'      1885
''           2
'Null'       1
Name: count, dtype: int64

Righe con Hiring diverso da SI/NO:
        ID OVERALL Hiring
97      98      10   Null
813    816               
3852  3859      10       


### 0.6 ID e DURATA

In [7]:
ids = df_raw["ID"].astype(int)        # sicuro: verificato in 0.3 che è sempre intero
mancanti = sorted(set(range(1, ids.max() + 1)) - set(ids))
print(f"ID: n={len(ids)}, distinti={ids.nunique()}, min={ids.min()}, max={ids.max()}, "
      f"ordinati={ids.is_monotonic_increasing}, buchi nella sequenza={mancanti}")
assert ids.is_unique

dur = df_raw["DURATA"].astype(int)
print("\nDURATA (giorni) – statistiche descrittive, n =", dur.notna().sum())
print(dur.describe(percentiles=[.01, .05, .25, .5, .75, .95, .99]).round(1).to_string())

# Valori molto grandi: sono plausibili come seriali di data Excel? (solo diagnosi, nessuna correzione)
grandi = df_raw.loc[dur > 400, ["ID", "DURATA", "NAZIONE", "CORSO"]].copy()
grandi["se_fosse_data_excel"] = pd.to_datetime(grandi["DURATA"].astype(int), unit="D",
                                               origin="1899-12-30").dt.date
grandi

ID: n=4382, distinti=4382, min=1, max=4388, ordinati=True, buchi nella sequenza=[266, 378, 1918, 2500, 3093, 3443]

DURATA (giorni) – statistiche descrittive, n = 4382
count     4382.0
mean       179.9
std       1812.5
min         20.0
1%          30.0
5%          39.0
25%         60.0
50%         91.0
75%        171.0
95%        182.0
99%        344.8
max      45534.0


,ID,DURATA,NAZIONE,CORSO,se_fosse_data_excel
17,18,547,SERBIA,B12,1901-06-30
106,107,45303,FRANCIA,B03,2024-01-12
507,510,45394,ITALIA,B16,2024-04-12
940,943,45485,ITALIA,B01,2024-07-12
1079,1082,45504,ITALIA,B01,2024-07-31
1282,1285,45492,ITALIA,B10,2024-07-19
1891,1894,45527,ITALIA,B11,2024-08-23
2302,2306,729,ITALIA,B17,1901-12-29
3139,3145,45534,GRAN BRETAGNA E IRLANDA DEL NORD,T07,2024-08-30


### 0.7 Anomalie puntuali e duplicati (solo elenco, nessuna correzione)

In [8]:
num_scores = df_raw[SCORES].apply(pd.to_numeric, errors="coerce")

# Righe con almeno una cella vuota
righe_vuote = df_raw[(df_raw == "").any(axis=1)]
print(f"Righe con almeno una cella vuota: n={len(righe_vuote)}")
print(righe_vuote[["ID", "DURATA", "NAZIONE", "AREA FUNZIONALE", "CORSO", "OVERALL", "Hiring"]], "\n")

# Punteggi fuori scala (numerici fuori 1-10) o token non numerici diversi da N/A
fuori = (num_scores < 1) | (num_scores > 10)
token = num_scores.isna() & ~df_raw[SCORES].isin(["", "N/A"])
anom = df_raw.loc[(fuori | token).any(axis=1), ["ID"] + SCORES]
print(f"Righe con punteggi fuori scala o token anomali: n={len(anom)}")
print(anom.to_string(index=False), "\n")

# N/A letterali per riga
na_per_riga = (df_raw[SCORES] == "N/A").sum(axis=1)
print("Distribuzione n. di N/A per riga:", na_per_riga.value_counts().sort_index().to_dict())
print("Righe con almeno un N/A:", int((na_per_riga > 0).sum()), "\n")

# Duplicati: identici su tutte le colonne tranne ID
dup_mask = df_raw.drop(columns="ID").duplicated(keep=False)
print(f"Duplicati esatti (escluso ID): righe coinvolte={dup_mask.sum()}, "
      f"copie in eccesso={df_raw.drop(columns='ID').duplicated().sum()}")
print("di cui con tutti i 17 punteggi = 10:", int((df_raw.loc[dup_mask, SCORES] == "10").all(axis=1).sum()))
print("Righe totali con tutti i 17 punteggi = 10:", int((df_raw[SCORES] == "10").all(axis=1).sum()))

pd.concat([
    anom.assign(tipo="punteggio fuori scala / token")[["ID", "tipo"]],
    righe_vuote.assign(tipo="cella vuota")[["ID", "tipo"]],
    df_raw.loc[dup_mask].assign(tipo="duplicato escluso ID")[["ID", "tipo"]],
    df_raw.loc[dur > 400].assign(tipo="DURATA > 400")[["ID", "tipo"]],
]).sort_values("ID").to_csv(OUT_TAB / "step0_righe_anomale.csv", index=False)

Righe con almeno una cella vuota: n=4
        ID DURATA NAZIONE                       AREA FUNZIONALE CORSO OVERALL Hiring
813    816    330                                                                   
1131  1134    212                            Ricerca & Sviluppo   B01      10     NO
2027  2031     85                       Attività polifunzionali   T01      10     SI
3852  3859    195  ITALIA  Amministrazione, Finanza e Controllo   M03      10        

Righe con punteggi fuori scala o token anomali: n=8
  ID Digital Teamwork Entrepreneurial Mindset Critical Thinking Creativity Planning Flexibility Cultural Awareness Writing Listening Time Management Responsibility Motivation Feedback Management Interpersonal Skills Leadership OVERALL
   1      10       10                      10                10         10       10          10                  A      10        10              10             10         10                  10                   10         10      10
  42      10 

### 0.8 Verifica indipendente: CSV vs XLSX

In [9]:
if DATA_XLSX.exists():
    xl = pd.read_excel(DATA_XLSX, sheet_name=None, dtype=str, keep_default_na=False)
    print("Fogli:", {k: v.shape for k, v in xl.items()})
    x = next(iter(xl.values()))
    assert list(x.columns) == list(df_raw.columns) and x.shape == df_raw.shape
    n_diff = int((x.values != df_raw.values).sum())
    print("Celle diverse tra CSV e XLSX:", n_diff)
    assert n_diff == 0
else:
    print("XLSX non presente: controllo saltato")

Fogli: {'in': (4382, 24)}
Celle diverse tra CSV e XLSX: 0


### Registro assunzioni e limiti – aggiornato a fine Step 0

| # | Tipo | Voce | Stato |
|---|------|------|-------|
| A1 | Assunzione | Codifica `cp1252` (unico byte non ASCII: `0xE0` = "à"); separatore `;`; fine riga CRLF. | verificata |
| A2 | Assunzione | `NAZIONE` = paese della sede dello stage, non nazionalità dello studente (da brief). | da brief |
| A3 | Assunzione | `CORSO` trattato solo come codice anonimo; nessuna ipotesi sul tipo di corso. | da brief |
| A4 | Assunzione | Scala attesa dei punteggi: interi 1-10. | da brief, coerente con 99,9% dei valori |
| L1 | Qualità | `N/A` letterale in 15 skill su 16 (non in `Leadership` né in `OVERALL`); 470 righe con almeno un N/A. Significato (non osservato? non applicabile?) ignoto. | da decidere |
| L2 | Qualità | 5 valori fuori scala (7000, 88, 999, 55, 100) e 3 token non numerici (`A`, `BBBB`, `NN`). | da decidere |
| L3 | Qualità | Righe con celle vuote: ID 816 (quasi vuota), 1134 e 2031 (NAZIONE vuota), 3859 (Hiring vuoto); ID 98 ha Hiring = `Null`. | da decidere |
| L4 | Qualità | `DURATA`: 7 valori tra 45.303 e 45.534 compatibili con date Excel 2024; 2 valori (547, 729) alti ma non impossibili. | da decidere |
| L5 | Qualità | 9 coppie identiche su tutte le colonne tranne ID; 8 su 9 sono valutazioni "tutto 10", che nel file sono 742: potrebbero essere coincidenze. | da decidere |
| L6 | Qualità | `SETTORE AZIENDALE` 'ONG ed Enti No Profit ' ha uno spazio finale (unica categoria, nessuno sdoppiamento). | da decidere (cosmetico) |
| L7 | Qualità | `CORSO` = 'L' (n=314) ha formato diverso dagli altri codici (lettera + 2 cifre). | da chiarire |
| L8 | Limite | Forte sbilanciamento: ITALIA 3.403 righe su 4.382; molti paesi, corsi e aree con n molto piccolo. | da gestire in analisi |
| L9 | Limite | ID non contigui (mancano 6 ID): possibile pre-filtraggio a monte, ma nessun duplicato di ID. | informativo |

## Step 1 – Audit di qualità del dato (solo diagnosi, nessun trattamento)

**Regola dello step:** `df_raw` non viene mai modificato. Tutti i calcoli usano *viste temporanee*
(maschere booleane e conversioni numeriche in variabili separate). All'inizio e alla fine dello step
`df_raw` viene confrontato cella per cella con il file riletto da zero.

**Approccio e alternative considerate**

| Controllo | Scelta | Alternativa scartata e perché |
|---|---|---|
| Celle dei voti | Ogni cella in **una sola** categoria: valido 1-10, `N/A`, vuoto, testo non numerico, decimale, fuori scala (assert di partizione) | Conversione diretta a numerico: confonde `N/A`, vuoti e refusi in un unico `NaN` |
| Duplicati | Per ID, per riga intera escluso ID, ed escluso anche Hiring; stima di quante coincidenze ci si aspetta per caso | Duplicati "quasi uguali": senza un identificativo dello studente non si distinguono da stage diversi |
| Etichette | Normalizzazione + similarità testuale + revisione delle convenzioni di nome; plausibilità di SETTORE con V di Cramér e p-value per permutazione | Correzione automatica delle varianti: rischia di unire categorie diverse. Chi-quadro asintotico: non valido con molte celle attese < 5 |
| DURATA | Soglie esplicite e motivate (< 30 gg; > 400 gg; seriali Excel 2000-2030) + controllo dei valori ripetuti; regola IQR solo come riferimento | Solo IQR: la distribuzione è asimmetrica e segnalerebbe come anomali stage normali di 11-12 mesi |
| Mancanti casuali? | Tasso di Hiring con e senza skill mancanti (Fisher/chi-quadro, IC di Wilson), test di tendenza, Holm per le 15 skill, regressione logistica aggiustata per area e corso | Test di Little (MCAR): non disponibile nelle librerie concordate e presuppone normalità multivariata, inadatta a voti 1-10 molto asimmetrici |
| OVERALL vs skill | Spearman (principale) e Pearson; scarto OVERALL − media skill su 3 campioni; quota di OVERALL uguale alla media arrotondata | Solo Pearson: gonfiato dalle molte valutazioni "tutto 10" |
| Valutazioni piatte | 16 skill valide e identiche; a parte le piatte con 8-15 skill valide | Deviazione standard ≤ soglia: soglia arbitraria |

Un test può solo **trovare evidenza contro** la casualità dei mancanti, mai dimostrarla; con questi dati non si
distingue tra mancanti legati a variabili osservate (MAR) e legati al voto mancante stesso (MNAR).

In [10]:
# --- Step 1: setup dell'audit (nessuna modifica a df_raw) -------------------------------
import re, unicodedata, difflib
from math import comb
from itertools import combinations
from scipy import stats
from statsmodels.stats.proportion import proportion_confint
from statsmodels.stats.multitest import multipletests
import statsmodels.formula.api as smf

def rileggi_file():
    """Rilegge il CSV da zero, con le stesse opzioni dello Step 0."""
    return pd.read_csv(DATA_CSV, sep=";", encoding=ENCODING, dtype=str, keep_default_na=False)
pd.testing.assert_frame_equal(df_raw, rileggi_file())     # punto di partenza: df_raw == file

N_TOT    = len(df_raw)
LABELS   = ["NAZIONE", "SETTORE AZIENDALE", "AREA FUNZIONALE", "CORSO"]
SOGLIA_N = 30                     # numerosità minima concordata per mostrare un segmento
SOGLIA_DURATA_BASSA = 30          # meno di un mese
SOGLIA_DURATA_ALTA  = 400         # oltre ~13 mesi (nel file non ci sono valori tra 390 e 546)
RNG = np.random.default_rng(2026) # seme fisso: i test di permutazione sono riproducibili

def pct(n, den=N_TOT):
    """Percentuale con 2 decimali (NaN se il denominatore è 0)."""
    return round(100 * n / den, 2) if den else np.nan

def n_pct(n, den=N_TOT):
    """Formato 'n (x% di den)' per riportare sempre la numerosità."""
    return f"{int(n)} ({pct(n, den):.2f}% di {int(den)})"

def ids(mask):
    """Elenco degli ID selezionati da una maschera."""
    return df_raw.loc[mask, "ID"].tolist()

def chi2_perm(a, b, n_perm=2000):
    """Indipendenza tra due variabili categoriche: chi-quadro, p asintotico, p per permutazione
    (valido anche con celle attese < 5), V di Cramér e n."""
    ca, _ = pd.factorize(np.asarray(a)); cb, _ = pd.factorize(np.asarray(b))
    ka, kb = ca.max() + 1, cb.max() + 1
    def chi2_di(cb_):
        oss = np.bincount(ca * kb + cb_, minlength=ka * kb).reshape(ka, kb).astype(float)
        att = oss.sum(1, keepdims=True) * oss.sum(0, keepdims=True) / oss.sum()
        return ((oss - att) ** 2 / att).sum(), (att < 5).mean()
    chi2, quota_att_lt5 = chi2_di(cb)
    perm = np.array([chi2_di(RNG.permutation(cb))[0] for _ in range(n_perm)])
    n = len(ca)
    return {"n": n, "chi2": round(chi2, 1), "gdl": (ka - 1) * (kb - 1),
            "p_asintotico": round(stats.chi2.sf(chi2, (ka - 1) * (kb - 1)), 4),
            "p_permutazione": round((1 + (perm >= chi2).sum()) / (n_perm + 1), 4),
            "V_cramer": round(np.sqrt(chi2 / (n * (min(ka, kb) - 1))), 3),
            "pct_celle_attese_lt5": round(100 * quota_att_lt5, 1)}

# Tabella riassuntiva: una riga per problema; le maschere servono per il conteggio dell'unione
audit, maschere_audit = [], {}
def registra(problema, colonne, mask_righe, trattamento, rischio, tipo, stato):
    """tipo: 'errore da correggere' | 'limite da dichiarare' | 'nessun problema';
    stato: 'già deciso' | 'da decidere'. n e % sono calcolati dalla maschera."""
    mask = pd.Series(np.asarray(mask_righe, dtype=bool), index=df_raw.index)
    n = int(mask.sum())
    audit.append({"problema": problema, "colonne": colonne, "n_righe": n, "pct_sul_totale": pct(n),
                  "trattamento_proposto": trattamento, "rischio_del_trattamento": rischio,
                  "tipo": tipo, "stato": stato})
    maschere_audit[problema] = (tipo, mask)

### 1.1 Skill e OVERALL: valori non numerici, mancanti, fuori scala

In [11]:
txt = df_raw[SCORES]                                    # testo originale
num = txt.apply(pd.to_numeric, errors="coerce")         # vista numerica temporanea (NaN se non numero)

m_na     = txt.eq("N/A")                                # mancante dichiarato
m_vuoto  = txt.eq("")                                   # cella vuota
m_token  = num.isna() & ~m_na & ~m_vuoto                # testo non numerico (es. 'A', 'BBBB')
in_scala = (num >= 1) & (num <= 10)
m_fuori  = num.notna() & ~in_scala                      # numero fuori dalla scala 1-10
m_dec    = in_scala & (num % 1 != 0)                    # numero nella scala ma con decimali
m_valido = in_scala & (num % 1 == 0)                    # voto valido

# Controllo automatico: ogni cella appartiene a una e una sola categoria
partizione = sum(m.astype(int) for m in [m_na, m_vuoto, m_token, m_fuori, m_dec, m_valido])
assert (partizione == 1).all().all(), "classificazione delle celle non esclusiva"
assert m_dec.values.sum() == 0, "presenti voti decimali: da esaminare"

def elenca(mask, col):
    """Elenco 'ID: valore' delle celle segnalate in una colonna."""
    return "; ".join(f"ID {i}: {v!r}" for i, v in zip(df_raw.loc[mask[col], "ID"], txt.loc[mask[col], col]))

qual_voti = pd.DataFrame({
    "n_validi": m_valido.sum(), "n_NA": m_na.sum(), "n_vuoti": m_vuoto.sum(),
    "n_testo_non_numerico": m_token.sum(), "n_decimali": m_dec.sum(), "n_fuori_scala": m_fuori.sum(),
}).astype(int)
qual_voti["pct_non_validi"] = (100 * (N_TOT - qual_voti["n_validi"]) / N_TOT).round(2)
qual_voti["celle_anomale"] = [elenca(m_token | m_fuori, c) for c in SCORES]
qual_voti.loc["TOTALE (celle)"] = list(qual_voti.iloc[:, :6].sum()) + [np.nan, ""]
qual_voti.to_csv(OUT_TAB / "step1_qualita_voti.csv")
display(qual_voti)

# Livello riga
n_sk     = m_valido[SKILLS].sum(axis=1)                              # skill valide per riga
r_vuota  = df_raw.drop(columns="ID").eq("").sum(axis=1) >= 10        # riga (quasi) vuota
r_token  = m_token.any(axis=1)
r_fuori  = m_fuori.any(axis=1)
r_na     = m_na.any(axis=1)
r_una_sk = (n_sk <= 1) & ~r_vuota                                    # valutazione con al massimo 1 skill valida
assert m_vuoto[~r_vuota].values.sum() == 0, "celle vuote nei voti fuori dalla riga quasi vuota"
print("Righe con testo non numerico:", n_pct(r_token.sum()), "-> ID", ids(r_token))
print("Righe con valori fuori scala:", n_pct(r_fuori.sum()), "-> ID", ids(r_fuori))
print("Righe con almeno un N/A    :", n_pct(r_na.sum()), "| celle N/A:", int(m_na.values.sum()))
print("Righe quasi vuote (>= 10 celle vuote):", n_pct(r_vuota.sum()), "-> ID", ids(r_vuota),
      "| celle vuote:", int(df_raw.loc[r_vuota].drop(columns="ID").eq("").sum(axis=1).iloc[0]), "su 23")
print("Valutazioni con <= 1 skill valida (esclusa la riga vuota):", n_pct(r_una_sk.sum()), "-> ID", ids(r_una_sk))

,n_validi,n_NA,n_vuoti,n_testo_non_numerico,n_decimali,n_fuori_scala,pct_non_validi,celle_anomale
Digital,4316,65,1,0,0,0,1.51,
Teamwork,4291,90,1,0,0,0,2.08,
Entrepreneurial Mindset,4227,153,1,0,0,1,3.54,ID 42: '7000'
Critical Thinking,4254,127,1,0,0,0,2.92,
Creativity,4285,96,1,0,0,0,2.21,
Planning,4286,95,1,0,0,0,2.19,
Flexibility,4369,12,1,0,0,0,0.30,
Cultural Awareness,4330,49,1,1,0,1,1.19,ID 1: 'A'; ID 3919: '88'
Writing,4375,5,1,0,0,1,0.16,ID 3863: '999'
Listening,4377,3,1,1,0,0,0.11,ID 4378: 'BBBB'


Righe con testo non numerico: 3 (0.07% di 4382) -> ID ['1', '475', '4378']
Righe con valori fuori scala: 5 (0.11% di 4382) -> ID ['42', '73', '3811', '3863', '3919']
Righe con almeno un N/A    : 470 (10.73% di 4382) | celle N/A: 772
Righe quasi vuote (>= 10 celle vuote): 1 (0.02% di 4382) -> ID ['816'] | celle vuote: 21 su 23
Valutazioni con <= 1 skill valida (esclusa la riga vuota): 2 (0.05% di 4382) -> ID ['2482', '3563']


In [12]:
# Fuori scala: proposta di ricodifica secondo una regola esplicita (solo proposta, NON applicata).
# Regola concordata: cifra ripetuta ('55', '88', '999') -> la cifra; '100' -> 10; altrimenti -> mancante.
def proposta_refuso(v):
    if v == "100":
        return 10
    if len(v) >= 2 and len(set(v)) == 1 and v[0] in "123456789":
        return int(v[0])
    return None

righe = []
for col in SCORES:
    for idx in df_raw.index[m_fuori[col]]:
        altre = num.loc[idx, SKILLS].drop(col, errors="ignore").where(m_valido.loc[idx, SKILLS].drop(col, errors="ignore"))
        prop = proposta_refuso(txt.at[idx, col])
        righe.append({"ID": df_raw.at[idx, "ID"], "colonna": col, "valore": txt.at[idx, col],
                      "skill_valide_altre_n": int(altre.notna().sum()), "altre_min": altre.min(),
                      "altre_mediana": altre.median(), "altre_max": altre.max(),
                      "proposta": prop if prop is not None else "mancante"})
fuori_scala = pd.DataFrame(righe).sort_values("ID")
# Controllo di plausibilità: ogni valore ricodificato cade tra il minimo e il massimo delle altre skill della riga
ric = fuori_scala[fuori_scala.proposta.ne("mancante")]
assert ((ric.proposta.astype(float) >= ric.altre_min) & (ric.proposta.astype(float) <= ric.altre_max)).all()
fuori_scala.to_csv(OUT_TAB / "step1_fuori_scala.csv", index=False)
display(fuori_scala)

r_ricod     = df_raw["ID"].isin(ric.ID)
r_non_ricod = df_raw["ID"].isin(fuori_scala.loc[fuori_scala.proposta.eq("mancante"), "ID"])
cols_di = lambda mask: ", ".join(c for c in SCORES if mask[c].any())
skill_con_na = [c for c in SKILLS if m_na[c].any()]

registra("Testo non numerico nei voti", cols_di(m_token), r_token,
         "Convertire in mancante solo per quella skill", "Minimo: il voto reale è irrecuperabile",
         "errore da correggere", "già deciso")
registra("Fuori scala ricodificabile (" + ", ".join(f"{v}→{p}" for v, p in zip(ric.valore, ric.proposta)) + ")",
         ", ".join(ric.colonna), r_ricod,
         "Ricodificare con la regola 'cifra ripetuta → cifra; 100 → 10'",
         "Il valore reale potrebbe essere diverso (999 è spesso usato come codice di 'mancante')",
         "errore da correggere", "già deciso")
registra("Fuori scala non ricodificabile (" + ", ".join(fuori_scala.loc[fuori_scala.proposta.eq('mancante'), 'valore']) + ")",
         ", ".join(fuori_scala.loc[fuori_scala.proposta.eq("mancante"), "colonna"]), r_non_ricod,
         "Convertire in mancante solo per quella skill", "Minimo: 1 cella", "errore da correggere", "già deciso")
registra("Riga quasi vuota", "tutte tranne " + ", ".join(c for c in df_raw.columns if df_raw.loc[r_vuota, c].ne("").all()),
         r_vuota, "Escludere la riga", "Nessuno rilevante: non contiene valutazioni", "errore da correggere", "già deciso")
registra("Valutazione con 1 sola skill valida (15 N/A)", "15 skill", r_una_sk,
         "Tenere per OVERALL e Hiring; escludere dai confronti tra skill e dalla coerenza OVERALL-skill",
         "Nessuno rilevante (2 righe)", "limite da dichiarare", "da decidere")

,ID,colonna,valore,skill_valide_altre_n,altre_min,altre_mediana,altre_max,proposta
4,3811,OVERALL,100,16,9.0,10.0,10.0,10
2,3863,Writing,999,15,7.0,9.0,10.0,9
1,3919,Cultural Awareness,88,15,5.0,6.0,8.0,8
0,42,Entrepreneurial Mindset,7000,15,10.0,10.0,10.0,mancante
3,73,Responsibility,55,15,4.0,6.0,7.0,5


### 1.2 Duplicati

In [13]:
dup_id   = df_raw["ID"].duplicated(keep=False)
senza_id = df_raw.drop(columns="ID")
dup_riga = senza_id.duplicated(keep=False)

# Verifica indipendente: groupby su tutte le colonne tranne ID
gruppi = df_raw.groupby(list(senza_id.columns), sort=False)["ID"].agg(list)
gruppi = gruppi[gruppi.str.len() > 1]
assert gruppi.str.len().sum() == dup_riga.sum()

def descrivi(chiave, id_list):
    r = dict(zip(senza_id.columns, chiave))
    voti = {r[c] for c in SKILLS}
    return {"ID": ", ".join(id_list), "voto_unico_16_skill": voti.pop() if len(voti) == 1 else "-",
            "OVERALL": r["OVERALL"], "DURATA": r["DURATA"], "NAZIONE": r["NAZIONE"],
            "CORSO": r["CORSO"], "Hiring": r["Hiring"]}
coppie = pd.DataFrame([descrivi(k, v) for k, v in gruppi.items()])
coppie.to_csv(OUT_TAB / "step1_righe_identiche.csv", index=False)
print("Duplicati di ID:", n_pct(dup_id.sum()))
print("Righe identiche escluso ID:", n_pct(dup_riga.sum()), f"in {len(gruppi)} gruppi")
display(coppie)

# Identiche escludendo anche Hiring: coppie con stesso profilo ma esito di Hiring opposto
cols_no_h = list(senza_id.columns.drop("Hiring"))
dup_no_h = df_raw.duplicated(cols_no_h, keep=False)
g_no_h = df_raw[dup_no_h].groupby(cols_no_h, sort=False).agg(ID=("ID", list), Hiring=("Hiring", list))
opposti = g_no_h[g_no_h.Hiring.map(lambda h: len(set(h)) > 1)]
print(f"\nIdentiche escluso ID e Hiring: {n_pct(dup_no_h.sum())} in {len(g_no_h)} gruppi; "
      f"di cui {len(opposti)} con Hiring opposto:")
display(opposti.reset_index(drop=True))

# Quante coppie identiche ci aspetteremmo per puro caso?
# Coppie che coincidono su tutto tranne ID e SETTORE; se SETTORE è indipendente dal resto (vedi 1.3),
# la probabilità che due righe abbiano anche lo stesso settore è sum(p_settore^2).
cols_no_s = list(senza_id.columns.drop("SETTORE AZIENDALE"))
coppie_no_s = sum(comb(k, 2) for k in df_raw.groupby(cols_no_s).size())
p_sett = df_raw["SETTORE AZIENDALE"].value_counts(normalize=True)
attese = coppie_no_s * (p_sett ** 2).sum()
osservate = sum(comb(len(v), 2) for v in gruppi.values)
p_poisson = stats.poisson.sf(osservate - 1, attese)
print(f"Coppie uguali su tutto tranne ID e SETTORE: {coppie_no_s}; coppie identiche attese per caso: {attese:.2f}; "
      f"osservate: {osservate}; P(>= {osservate} | caso) = {p_poisson:.2f}")

registra("ID duplicato", "ID", dup_id, "Nessuno", "-", "nessun problema", "già deciso")
registra(f"Righe identiche su tutte le colonne tranne ID ({len(gruppi)} coppie)", "tutte tranne ID", dup_riga,
         "Nessuno: ID diverso = stage diverso",
         f"Se fossero doppi inserimenti, {len(gruppi)} valutazioni conterebbero due volte (attese per caso {attese:.1f}, osservate {osservate})",
         "limite da dichiarare", "già deciso")

Duplicati di ID: 0 (0.00% di 4382)
Righe identiche escluso ID: 18 (0.41% di 4382) in 9 gruppi


,ID,voto_unico_16_skill,OVERALL,DURATA,NAZIONE,CORSO,Hiring
0,"216, 954",10,10,179,ITALIA,B01,SI
1,"865, 916",9,9,95,ITALIA,T01,SI
2,"1052, 1369",10,10,178,FRANCIA,B04,SI
3,"1065, 1211",10,10,90,ITALIA,T08,SI
4,"1381, 2388",10,10,179,ITALIA,B03,SI
5,"1786, 4161",10,10,39,ITALIA,T01,SI
6,"2208, 2714",10,10,53,ITALIA,B14,SI
7,"3217, 3602",10,10,41,ITALIA,T01,SI
8,"3391, 3397",10,10,109,ITALIA,B11,SI



Identiche escluso ID e Hiring: 24 (0.55% di 4382) in 12 gruppi; di cui 3 con Hiring opposto:


,ID,Hiring
0,"[257, 2844]","[SI, NO]"
1,"[1421, 4339]","[NO, SI]"
2,"[3306, 4237]","[NO, SI]"


Coppie uguali su tutto tranne ID e SETTORE: 233; coppie identiche attese per caso: 6.17; osservate: 9; P(>= 9 | caso) = 0.17


### 1.3 Etichette di NAZIONE, SETTORE AZIENDALE, AREA FUNZIONALE, CORSO

In [14]:
def normalizza(s):
    """Chiave di confronto: maiuscolo, senza accenti, solo lettere e cifre."""
    s = unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode()
    return re.sub(r"[^A-Z0-9]", "", s.upper())

sintesi_lab, simili = [], []
for c in LABELS:
    s = df_raw[c]
    vc = s[s != ""].value_counts()
    vals = pd.Series(vc.index)
    chiavi = vals.map(normalizza)
    sintesi_lab.append({
        "colonna": c, "n_etichette": len(vals),
        "n_righe_mancanti": int((s == "").sum()), "ID_mancanti": ids(s == ""),
        "n_righe_spazi_extra": int((s != s.str.strip()).sum()),
        "etichette_con_spazi": [v for v in vals if v != v.strip()],
        "varianti_stessa_chiave": sorted(vals[chiavi.duplicated(keep=False)].tolist()),
        "n_categorie_n_lt_30": int((vc < SOGLIA_N).sum()),
        "n_righe_in_categorie_n_lt_30": int(vc[vc < SOGLIA_N].sum()),
    })
    # coppie di etichette molto simili (da leggere a occhio: simili non significa uguali)
    for a, b in combinations(vals, 2):
        r = difflib.SequenceMatcher(None, normalizza(a), normalizza(b)).ratio()
        if r >= 0.85:
            simili.append({"colonna": c, "etichetta_1": a, "n_1": int(vc[a]),
                           "etichetta_2": b, "n_2": int(vc[b]), "similarita": round(r, 2)})
sintesi_lab = pd.DataFrame(sintesi_lab)
simili = pd.DataFrame(simili, columns=["colonna", "etichetta_1", "n_1", "etichetta_2", "n_2", "similarita"])
sintesi_lab.to_csv(OUT_TAB / "step1_etichette_sintesi.csv", index=False)
simili.to_csv(OUT_TAB / "step1_etichette_simili.csv", index=False)
display(sintesi_lab)
print("Coppie di etichette con similarità >= 0,85 (da leggere a occhio):")
display(simili)

# Convenzioni di nome in NAZIONE (forme invertite, parentesi, trattini) ed etichette ambigue
naz = df_raw["NAZIONE"].copy()
vc_naz = naz[naz != ""].value_counts()
conv = vc_naz[vc_naz.index.str.contains(r"\sREPUBBLICA(?:\sPOPOLARE)?$|\(|\s-\s", regex=True)]   # forme invertite, parentesi, trattini
print("NAZIONE – convenzioni di nome non uniformi:", conv.to_dict())
AMBIGUE = {"MONACO": "Principato di Monaco o Monaco di Baviera (Germania)?"}   # giudizio manuale, dichiarato
print("NAZIONE – etichette ambigue:", {k: (int(vc_naz.get(k, 0)), v) for k, v in AMBIGUE.items()})

# Etichette residuali (categorie "contenitore")
ar = df_raw["AREA FUNZIONALE"].copy()
res_area = ar[ar.str.contains(r"^(?:Altro|Generico)$|non in elenco", regex=True)].value_counts()
res_sett = df_raw["SETTORE AZIENDALE"][df_raw["SETTORE AZIENDALE"].str.contains(r"^(?:Altri|Altre)\b", regex=True)].value_counts()
print("AREA – etichette residuali:", res_area.to_dict(), "| totale righe:", int(res_area.sum()))
print("AREA – 'Attività polifunzionali' (trasversale, non residuale):", int(ar.eq("Attività polifunzionali").sum()))
print("SETTORE – categorie 'Altri/Altre':", res_sett.to_dict(), "| totale righe:", int(res_sett.sum()))

# Formato dei codici CORSO: atteso lettera + 2 cifre
co = df_raw["CORSO"].copy()
corso_fuori_formato = co.ne("") & ~co.str.fullmatch(r"[A-Z]\d{2}")
legale = ar.str.startswith("Legale")
for cod in co[corso_fuori_formato].unique():
    m = co.eq(cod)
    print(f"CORSO fuori formato '{cod}': n={int(m.sum())}; con area Legale {n_pct((m & legale).sum(), m.sum())} "
          f"vs altri codici {n_pct((~m & co.ne('') & legale).sum(), (~m & co.ne('')).sum())}")

,colonna,n_etichette,n_righe_mancanti,ID_mancanti,n_righe_spazi_extra,etichette_con_spazi,varianti_stessa_chiave,n_categorie_n_lt_30,n_righe_in_categorie_n_lt_30
0,NAZIONE,64,3,"[816, 1134, 2031]",0,[],[],54,294
1,SETTORE AZIENDALE,38,0,[],131,[ONG ed Enti No Profit ],[],0,0
2,AREA FUNZIONALE,22,1,[816],0,[],[],7,137
3,CORSO,37,1,[816],0,[],[],13,95


Coppie di etichette con similarità >= 0,85 (da leggere a occhio):


,colonna,etichetta_1,n_1,etichetta_2,n_2,similarita
0,NAZIONE,AUSTRIA,6,AUSTRALIA,5,0.88


NAZIONE – convenzioni di nome non uniformi: {'CINA REPUBBLICA POPOLARE': 36, 'CECA REPUBBLICA': 4, 'IRLANDA - EIRE': 2, 'SUDAFRICANA REPUBBLICA': 1, 'DOMINICANA REPUBBLICA': 1, 'RUSSIA - FEDERAZIONE RUSSA': 1, 'CAYMAN (ISOLE)': 1}
NAZIONE – etichette ambigue: {'MONACO': (4, 'Principato di Monaco o Monaco di Baviera (Germania)?')}
AREA – etichette residuali: {'Legale - Ulteriori aree non in elenco': 47, 'Generico': 18, 'Altro': 3} | totale righe: 68
AREA – 'Attività polifunzionali' (trasversale, non residuale): 217
SETTORE – categorie 'Altri/Altre': {'Altre industrie manufatturiere': 131, 'Altri enti locali, amministrazioni centrali, authority': 120, 'Altri servizi alle imprese e alla persona': 109, 'Altri studi professionali': 100} | totale righe: 460
CORSO fuori formato 'L': n=314; con area Legale 287 (91.40% di 314) vs altri codici 68 (1.67% di 4067)


In [15]:
# Plausibilità di SETTORE AZIENDALE
# (a) distribuzione tra i 38 settori
vc_sett = df_raw["SETTORE AZIENDALE"].value_counts()
chi_unif = stats.chisquare(vc_sett.values)
print(f"SETTORE: {len(vc_sett)} settori, n per settore da {vc_sett.min()} a {vc_sett.max()} (n={vc_sett.sum()}); "
      f"test di uniformità chi2={chi_unif.statistic:.1f}, p={chi_unif.pvalue:.3f} "
      "(un p alto NON dimostra l'uniformità: dice solo che non la possiamo escludere)")

# (b) associazione con le altre dimensioni (V di Cramér, p per permutazione); controllo positivo AREA x CORSO
area_gr = ar.where(~legale, "Legale")                     # raggruppamento concordato (solo vista)
naz_b = naz.where(naz.isin(vc_naz[vc_naz >= SOGLIA_N].index), "ALTRI ESTERI").where(naz.ne(""), "")
ok = ar.ne("") & co.ne("") & naz.ne("") & df_raw["Hiring"].isin(["SI", "NO"])
sett = df_raw["SETTORE AZIENDALE"].str.strip()
assoc = pd.DataFrame([
    {"coppia": "SETTORE x AREA (Legale raggruppato)", **chi2_perm(sett[ok], area_gr[ok])},
    {"coppia": "SETTORE x CORSO",                    **chi2_perm(sett[ok], co[ok])},
    {"coppia": "SETTORE x NAZIONE (opzione B)",      **chi2_perm(sett[ok], naz_b[ok])},
    {"coppia": "SETTORE x Hiring",                   **chi2_perm(sett[ok], df_raw["Hiring"][ok])},
    {"coppia": "controllo positivo: AREA x CORSO",   **chi2_perm(area_gr[ok], co[ok])},
])
# Verifica indipendente del chi-quadro con scipy
chi_sc = stats.chi2_contingency(pd.crosstab(sett[ok], area_gr[ok]), correction=False)
assert np.isclose(chi_sc.statistic, assoc.chi2.iloc[0], atol=0.05)
assoc.to_csv(OUT_TAB / "step1_settore_associazioni.csv", index=False)
display(assoc)

# (c) coerenza "di buon senso" tra settore e area (pochi casi per settore: potenza bassa, contano le stime)
CONTROLLI = [("Studi legali e notarili", legale, "Legale (tutte le sottovoci)"),
             ("Informatica e attività connesse", ar.eq("Sistemi Informativi"), "Sistemi Informativi"),
             ("Revisione contabile", ar.eq("Amministrazione, Finanza e Controllo"), "Amministrazione, Finanza e Controllo"),
             ("Studi commercialisti, tributaristi ed esperti contabili", ar.eq("Amministrazione, Finanza e Controllo"), "Amministrazione, Finanza e Controllo")]
righe = []
con_area = ar.ne("")
for s_nome, m_area, a_nome in CONTROLLI:
    dentro, fuori = con_area & sett.eq(s_nome), con_area & ~sett.eq(s_nome)
    k_in, n_in, k_out, n_out = (dentro & m_area).sum(), dentro.sum(), (fuori & m_area).sum(), fuori.sum()
    p = stats.fisher_exact([[k_in, n_in - k_in], [k_out, n_out - k_out]])[1]
    righe.append({"settore": s_nome, "area_attesa": a_nome, "k_settore": int(k_in), "n_settore": int(n_in),
                  "pct_nel_settore": pct(k_in, n_in), "k_altri": int(k_out), "n_altri": int(n_out),
                  "pct_negli_altri": pct(k_out, n_out), "p_fisher": round(p, 3)})
coerenza_settore = pd.DataFrame(righe)
coerenza_settore.to_csv(OUT_TAB / "step1_coerenza_settore_area.csv", index=False)
display(coerenza_settore)

SETTORE: 38 settori, n per settore da 100 a 139 (n=4382); test di uniformità chi2=28.3, p=0.848 (un p alto NON dimostra l'uniformità: dice solo che non la possiamo escludere)


,coppia,n,chi2,gdl,p_asintotico,p_permutazione,V_cramer,pct_celle_attese_lt5
0,SETTORE x AREA (Legale raggruppato),4377,575.3,518,0.0409,0.0395,0.097,53.7
1,SETTORE x CORSO,4377,1336.7,1332,0.4587,0.4608,0.092,78.4
2,SETTORE x NAZIONE (opzione B),4377,351.0,370,0.7536,0.7416,0.090,81.8
3,SETTORE x Hiring,4377,46.3,37,0.1410,0.1454,0.103,0.0
4,controllo positivo: AREA x CORSO,4377,9280.8,504,0.0000,0.0005,0.389,69.9


,settore,area_attesa,k_settore,n_settore,pct_nel_settore,k_altri,n_altri,pct_negli_altri,p_fisher
0,Studi legali e notarili,Legale (tutte le sottovoci),8,113,7.08,347,4268,8.13,0.861
1,Informatica e attività connesse,Sistemi Informativi,3,106,2.83,134,4275,3.13,1.000
2,Revisione contabile,"Amministrazione, Finanza e Controllo",46,103,44.66,1677,4278,39.20,0.263
3,"Studi commercialisti, tributaristi ed esperti ...","Amministrazione, Finanza e Controllo",43,125,34.40,1680,4256,39.47,0.266


In [16]:
# Registrazione dei problemi di etichetta
vc_co = co[co != ""].value_counts()
vc_ar = area_gr[area_gr != ""].value_counts()
naz_manc = naz.eq("") & ~r_vuota
v_sa = assoc.set_index("coppia")

registra("NAZIONE mancante (righe altrimenti complete)", "NAZIONE", naz_manc,
         "Mancante solo per le analisi per paese", "Minimo", "errore da correggere", "già deciso")
registra("Spazio finale nell'etichetta " + ", ".join(repr(e) for e in sintesi_lab.etichette_con_spazi.sum()),
         "SETTORE AZIENDALE", df_raw["SETTORE AZIENDALE"].ne(df_raw["SETTORE AZIENDALE"].str.strip()),
         "Rimuovere gli spazi iniziali/finali", "Nessuno (nessuna categoria si fonde)",
         "errore cosmetico", "da decidere")
registra("Convenzioni di nome non uniformi (" + ", ".join(conv.index) + ")", "NAZIONE", naz.isin(conv.index),
         "Nessuna modifica (cosmetico: nessun paese compare con due nomi)", "Nessuno", "nessun problema", "da decidere")
registra("Etichetta ambigua (" + ", ".join(AMBIGUE) + ")", "NAZIONE", naz.isin(list(AMBIGUE)),
         "Nessuna modifica; chiarire se è il Principato di Monaco",
         f"Se fosse Monaco di Baviera, {int(naz.isin(list(AMBIGUE)).sum())} stage andrebbero sotto GERMANIA",
         "limite da dichiarare", "da decidere")
registra("Codice CORSO con formato diverso (" + ", ".join(co[corso_fuori_formato].unique()) + ")", "CORSO", corso_fuori_formato,
         "Nessuno: codice valido (coerente con le aree in cui compare)", "Nessuno rilevante",
         "nessun problema", "già deciso")
registra("Etichette residuali in AREA (" + ", ".join(res_area.index) + ")", "AREA FUNZIONALE", ar.isin(res_area.index),
         "Tenere; leggerle come non interpretabili ('Legale - Ulteriori…' confluisce in 'Legale')",
         "Un risultato su queste etichette non indica un'area precisa", "limite da dichiarare", "da decidere")
registra(f"Paesi con n<30 ({int((vc_naz < SOGLIA_N).sum())} paesi)", "NAZIONE", naz.isin(vc_naz[vc_naz < SOGLIA_N].index),
         "Raggruppare in 'Altri esteri' (opzione B)", "Gruppo eterogeneo: la media nasconde differenze tra paesi",
         "limite da dichiarare", "già deciso")
registra(f"Corsi con n<30 ({int((vc_co < SOGLIA_N).sum())} codici)", "CORSO", co.isin(vc_co[vc_co < SOGLIA_N].index),
         "Nessun accorpamento; esclusi dai confronti tra corsi, inclusi nei totali", "Nessuna informazione su questi corsi",
         "limite da dichiarare", "già deciso")
registra("Aree con n<30 dopo il raggruppamento 'Legale' (" + ", ".join(vc_ar[vc_ar < SOGLIA_N].index) + ")", "AREA FUNZIONALE",
         area_gr.isin(vc_ar[vc_ar < SOGLIA_N].index), "Escluse dai confronti tra aree, incluse nei totali",
         "Nessuna informazione su queste aree", "limite da dichiarare", "già deciso")
registra(f"SETTORE quasi indipendente da tutto (V={v_sa.V_cramer.iloc[0]} con l'area vs {v_sa.V_cramer.iloc[-1]} AREA x CORSO; "
         f"{vc_sett.min()}-{vc_sett.max()} righe per settore)", "SETTORE AZIENDALE", pd.Series(True, index=df_raw.index),
         "Non usarlo come dimensione di segmentazione; al più descrittivo, dichiarando il motivo",
         "Se il dato fosse corretto, si rinuncerebbe a una dimensione di lettura", "limite da dichiarare", "da decidere")

### 1.4 DURATA: valori implausibili, possibili date Excel, valori ripetuti

In [17]:
dur = pd.to_numeric(df_raw["DURATA"], errors="coerce")
assert dur.notna().all() and (dur > 0).all() and (dur % 1 == 0).all()

ORIGINE_EXCEL = pd.Timestamp("1899-12-30")                       # origine dei seriali di data Excel
SERIALE_MIN = (pd.Timestamp("2000-01-01") - ORIGINE_EXCEL).days   # 36526
SERIALE_MAX = (pd.Timestamp("2030-12-31") - ORIGINE_EXCEL).days
m_excel = dur.between(SERIALE_MIN, SERIALE_MAX)                    # compatibile con una data 2000-2030
m_alta  = (dur > SOGLIA_DURATA_ALTA) & ~m_excel
m_anno  = dur.between(366, SOGLIA_DURATA_ALTA)                     # poco oltre un anno
m_bassa = dur < SOGLIA_DURATA_BASSA
date_excel = ORIGINE_EXCEL + pd.to_timedelta(dur[m_excel], unit="D")
anni_excel = sorted(int(a) for a in date_excel.dt.year.unique())

d_ok = dur[~m_excel]
q1, q3 = d_ok.quantile([.25, .75]); soglia_iqr = q3 + 1.5 * (q3 - q1)
print(f"DURATA senza seriali Excel: n={len(d_ok)}, mediana={d_ok.median():.0f}, Q1={q1:.0f}, Q3={q3:.0f}, "
      f"soglia IQR={soglia_iqr:.1f} -> oltre soglia: {n_pct((d_ok > soglia_iqr).sum(), len(d_ok))} "
      f"(di cui entro 365 giorni: {int(((d_ok > soglia_iqr) & (d_ok <= 365)).sum())})")
print("Possibili date Excel:", n_pct(m_excel.sum()), "| anni:", anni_excel)
print(f"Oltre {SOGLIA_DURATA_ALTA} giorni:", n_pct(m_alta.sum()), dict(zip(ids(m_alta), dur[m_alta])))
print(f"Tra 366 e {SOGLIA_DURATA_ALTA} giorni (poco oltre un anno):", n_pct(m_anno.sum()), sorted(dur[m_anno]))
print(f"Sotto {SOGLIA_DURATA_BASSA} giorni:", n_pct(m_bassa.sum()), "| valori:", dur[m_bassa].value_counts().sort_index().to_dict())

# Valori molto ripetuti ("heaping"): possibili durate standard o predefinite
top = d_ok.value_counts().head(5).to_frame("n")
top["pct"] = (100 * top.n / len(d_ok)).round(2)
print(f"\n5 durate più frequenti (n={len(d_ok)}):"); display(top)
m_heap = dur.isin(top.index[:3])

durata_anomala = df_raw.loc[m_excel | m_alta | m_anno, ["ID", "DURATA", "NAZIONE", "CORSO"]].assign(
    tipo=np.select([m_excel, m_alta], ["possibile data Excel", f"oltre {SOGLIA_DURATA_ALTA} giorni"],
                   "366-400 giorni")[(m_excel | m_alta | m_anno).to_numpy()],
    data_se_seriale_excel=lambda d: d.DURATA.astype(int).map(
        lambda v: str((ORIGINE_EXCEL + pd.Timedelta(days=v)).date()) if SERIALE_MIN <= v <= SERIALE_MAX else ""))
durata_anomala.to_csv(OUT_TAB / "step1_durata_anomala.csv", index=False)
display(durata_anomala)

registra(f"DURATA compatibile con una data Excel ({', '.join(map(str, anni_excel))})", "DURATA", m_excel,
         "Convertire in mancante", "La durata reale è persa; nessuna ricostruzione possibile", "errore da correggere", "già deciso")
registra(f"DURATA oltre {SOGLIA_DURATA_ALTA} giorni ({', '.join(map(str, dur[m_alta]))})", "DURATA", m_alta,
         "Tenere e segnalare con un flag", "Valori rari ma possibili: usare la mediana per le durate",
         "limite da dichiarare", "già deciso")
registra(f"DURATA tra 366 e {SOGLIA_DURATA_ALTA} giorni", "DURATA", m_anno,
         "Tenere senza flag (stage di circa un anno)", "Nessuno rilevante", "nessun problema", "da decidere")
registra(f"DURATA sotto {SOGLIA_DURATA_BASSA} giorni", "DURATA", m_bassa, "Tenere: plausibile per stage brevi",
         "Nessuno rilevante", "nessun problema", "da decidere")
registra("DURATA concentrata su pochi valori (" + ", ".join(f"{v} gg: {n}" for v, n in top.n.iloc[:3].items()) + ")",
         "DURATA", m_heap, "Nessun trattamento; dichiarare che la durata può essere un valore standard e non quello effettivo",
         "Le analisi per durata vanno lette per fasce, non per giorno", "limite da dichiarare", "da decidere")

DURATA senza seriali Excel: n=4375, mediana=91, Q1=60, Q3=170, soglia IQR=335.0 -> oltre soglia: 39 (0.89% di 4375) (di cui entro 365 giorni: 30)
Possibili date Excel: 7 (0.16% di 4382) | anni: [2024]
Oltre 400 giorni: 2 (0.05% di 4382) {'18': 547, '2306': 729}
Tra 366 e 400 giorni (poco oltre un anno): 7 (0.16% di 4382) [366, 368, 373, 374, 387, 389, 389]
Sotto 30 giorni: 40 (0.91% di 4382) | valori: {20: 1, 23: 2, 24: 1, 25: 28, 27: 2, 28: 4, 29: 2}

5 durate più frequenti (n=4375):


,n,pct
DURATA,,
179,556,12.71
39,266,6.08
88,226,5.17
60,176,4.02
53,160,3.66


,ID,DURATA,NAZIONE,CORSO,tipo,data_se_seriale_excel
17,18,547,SERBIA,B12,oltre 400 giorni,
33,34,374,ITALIA,M13,366-400 giorni,
106,107,45303,FRANCIA,B03,possibile data Excel,2024-01-12
122,123,366,GERMANIA,B14,366-400 giorni,
128,129,389,ITALIA,T07,366-400 giorni,
507,510,45394,ITALIA,B16,possibile data Excel,2024-04-12
786,789,373,ITALIA,B16,366-400 giorni,
926,929,368,ITALIA,B01,366-400 giorni,
934,937,389,ITALIA,B04,366-400 giorni,
940,943,45485,ITALIA,B01,possibile data Excel,2024-07-12


### 1.5 Hiring: valori diversi da SI/NO

In [18]:
hir = df_raw["Hiring"].copy()
m_hir_bad = ~hir.isin(["SI", "NO"])
print("Hiring diverso da SI/NO:", n_pct(m_hir_bad.sum()))
display(df_raw.loc[m_hir_bad, ["ID", "OVERALL", "Hiring"]].assign(Hiring=lambda d: d.Hiring.map(repr)))
registra("Hiring vuoto o 'Null' (oltre alla riga quasi vuota)", "Hiring", m_hir_bad & ~r_vuota,
         "Mancante solo per le analisi su Hiring", "Minimo", "errore da correggere", "già deciso")

Hiring diverso da SI/NO: 3 (0.07% di 4382)


,ID,OVERALL,Hiring
97,98,10,'Null'
813,816,,''
3852,3859,10,''


### 1.6 Mancanti: distribuzione per skill e legame con Hiring (mancanti casuali o no?)
**Campione di tutti i confronti di questa sezione:** righe con Hiring = SI/NO (esclude anche la riga quasi vuota).
**Mancante** = `N/A` dichiarato. Testi non numerici e fuori scala sono *valori errati*, non mancanti, e non
rientrano in questa definizione (riguardano 8 celle).

In [19]:
camp = hir.isin(["SI", "NO"])
assert not (r_vuota & camp).any()                  # la riga quasi vuota è fuori campione
y    = hir.eq("SI")
m_na_sk = m_na[SKILLS]
k_miss  = m_na_sk.sum(axis=1)                      # numero di skill mancanti per riga
N_C = int(camp.sum())

# (a) distribuzione dei mancanti per skill (denominatore: tutte le righe; i vuoti sono tutti della riga quasi vuota)
miss_skill = pd.DataFrame({"n_NA": m_na_sk.sum(), "pct_NA": (100 * m_na_sk.sum() / N_TOT).round(2),
                           "n_vuoti": m_vuoto[SKILLS].sum(), "n_totale": N_TOT}).sort_values("n_NA", ascending=False)
miss_skill.to_csv(OUT_TAB / "step1_mancanti_per_skill.csv")
display(miss_skill)
print("Righe per numero di skill mancanti (tutte le righe):", k_miss.value_counts().sort_index().to_dict())

def tasso(mask):
    """n, n_SI, tasso di Hiring e IC di Wilson al 95% sul sottoinsieme mask (dentro il campione)."""
    m = mask & camp
    n, k = int(m.sum()), int((m & y).sum())
    lo, hi = proportion_confint(k, n, alpha=0.05, method="wilson")
    return {"n": n, "n_SI": k, "tasso_SI_pct": pct(k, n), "IC95_basso": round(100 * lo, 1), "IC95_alto": round(100 * hi, 1)}

# (b) Hiring: almeno una skill mancante vs nessuna
g1, g0 = tasso(k_miss > 0), tasso(k_miss == 0)
gruppi_miss = pd.DataFrame.from_dict({"≥1 skill mancante": g1, "nessuna skill mancante": g0}, orient="index")
display(gruppi_miss)
tab = np.array([[g1["n_SI"], g1["n"] - g1["n_SI"]], [g0["n_SI"], g0["n"] - g0["n_SI"]]])
chi2, p_chi2, _, _ = stats.chi2_contingency(tab, correction=False)
p_fisher = stats.fisher_exact(tab)[1]
# Verifica indipendente: chi-quadro calcolato a mano
att = tab.sum(1, keepdims=True) * tab.sum(0, keepdims=True) / tab.sum()
chi2_man = ((tab - att) ** 2 / att).sum()
assert np.isclose(chi2, chi2_man) and g1["n"] + g0["n"] == N_C
diff = 100 * (g1["n_SI"] / g1["n"] - g0["n_SI"] / g0["n"])
print(f"Differenza tasso SI (≥1 mancante − nessuno) = {diff:+.2f} punti (n={N_C}); chi2={chi2:.2f} (manuale {chi2_man:.2f}), "
      f"p={p_chi2:.4f}; Fisher p={p_fisher:.4f}")

# (c) tasso di Hiring per numero di skill mancanti (associazione) + test di tendenza (logit su 0,1,2,3+)
classe = pd.cut(k_miss, [-1, 0, 1, 2, 16], labels=["0", "1", "2", "3+"])
per_k = pd.DataFrame.from_dict({k: tasso(classe == k) for k in classe.cat.categories}, orient="index").rename_axis("n_skill_mancanti")
display(per_k)
d_tr = pd.DataFrame({"hiring": y.astype(int), "k": k_miss.clip(upper=3)})[camp]
tr = smf.logit("hiring ~ k", d_tr).fit(disp=0)
print(f"Tendenza (logit, k = 0..3+): OR per ogni skill mancante in più = {np.exp(tr.params['k']):.3f}, "
      f"p = {tr.pvalues['k']:.4f}, n = {int(tr.nobs)}")

,n_NA,pct_NA,n_vuoti,n_totale
Entrepreneurial Mindset,153,3.49,1,4382
Critical Thinking,127,2.90,1,4382
Creativity,96,2.19,1,4382
Planning,95,2.17,1,4382
Teamwork,90,2.05,1,4382
Digital,65,1.48,1,4382
Cultural Awareness,49,1.12,1,4382
Time Management,22,0.50,1,4382
Interpersonal Skills,20,0.46,1,4382
Flexibility,12,0.27,1,4382


Righe per numero di skill mancanti (tutte le righe): {0: 3912, 1: 295, 2: 110, 3: 43, 4: 13, 5: 1, 6: 3, 7: 2, 9: 1, 15: 2}


,n,n_SI,tasso_SI_pct,IC95_basso,IC95_alto
≥1 skill mancante,470,232,49.36,44.9,53.9
nessuna skill mancante,3909,2262,57.87,56.3,59.4


Differenza tasso SI (≥1 mancante − nessuno) = -8.50 punti (n=4379); chi2=12.38 (manuale 12.38), p=0.0004; Fisher p=0.0005


,n,n_SI,tasso_SI_pct,IC95_basso,IC95_alto
n_skill_mancanti,,,,,
0,3909,2262,57.87,56.3,59.4
1,295,160,54.24,48.5,59.8
2,110,43,39.09,30.5,48.4
3+,65,29,44.62,33.2,56.7


Tendenza (logit, k = 0..3+): OR per ogni skill mancante in più = 0.787, p = 0.0000, n = 4379


In [20]:
# (d) per singola skill: Hiring quando la skill è N/A vs quando è valida (correzione di Holm)
righe = []
for c in skill_con_na:
    a, b = tasso(m_na[c]), tasso(m_valido[c])
    p = stats.fisher_exact([[a["n_SI"], a["n"] - a["n_SI"]], [b["n_SI"], b["n"] - b["n_SI"]]])[1]
    righe.append({"skill": c, "n_NA": a["n"], "tasso_SI_NA": a["tasso_SI_pct"],
                  "n_valida": b["n"], "tasso_SI_valida": b["tasso_SI_pct"],
                  "diff_punti": round(a["tasso_SI_pct"] - b["tasso_SI_pct"], 2), "p_fisher": p})
hiring_per_skill = pd.DataFrame(righe)
hiring_per_skill["p_holm"] = multipletests(hiring_per_skill.p_fisher, method="holm")[1]
hiring_per_skill[["p_fisher", "p_holm"]] = hiring_per_skill[["p_fisher", "p_holm"]].round(4)
hiring_per_skill = hiring_per_skill.sort_values("diff_punti")
hiring_per_skill.to_csv(OUT_TAB / "step1_mancanti_vs_hiring_per_skill.csv", index=False)
display(hiring_per_skill)

# (e) OVERALL e DURATA: chi ha mancanti ha giudizi o durate diverse? (Mann-Whitney; stesso campione)
ov = num["OVERALL"].where(m_valido["OVERALL"])
du = dur.where(~m_excel)
righe = []
for nome, v in [("OVERALL", ov), ("DURATA (senza date Excel)", du)]:
    a, b = v[camp & (k_miss > 0) & v.notna()], v[camp & (k_miss == 0) & v.notna()]
    righe.append({"variabile": nome, "n_con_mancanti": len(a), "media_con": round(a.mean(), 2), "mediana_con": a.median(),
                  "n_senza": len(b), "media_senza": round(b.mean(), 2), "mediana_senza": b.median(),
                  "p_mann_whitney": round(stats.mannwhitneyu(a, b).pvalue, 4)})
display(pd.DataFrame(righe))

# (f) segmenti: la quota di righe con mancanti cambia tra paesi, aree, corsi? (categorie con n>=30, stesso campione)
righe = []
seg_it = pd.Series(np.where(naz.eq("ITALIA"), "ITALIA", np.where(naz.eq(""), "", "ESTERO")), index=df_raw.index)
for nome, s in [("NAZIONE (Italia/estero)", seg_it), ("NAZIONE (opzione B)", naz_b),
                ("AREA FUNZIONALE (Legale raggruppato)", area_gr), ("CORSO", co)]:
    vc = s[camp & s.ne("")].value_counts(); tenute = vc[vc >= SOGLIA_N].index
    m = camp & s.isin(tenute)
    ct = pd.crosstab(s[m], k_miss[m] > 0)
    quote = (100 * ct[True] / ct.sum(axis=1)).round(1)
    t = chi2_perm(s[m], k_miss[m] > 0)
    righe.append({"segmento": nome, "n": int(m.sum()), "n_categorie": len(tenute),
                  "min": f"{quote.idxmin()}: {quote.min()}% ({ct.loc[quote.idxmin(), True]}/{ct.loc[quote.idxmin()].sum()})",
                  "max": f"{quote.idxmax()}: {quote.max()}% ({ct.loc[quote.idxmax(), True]}/{ct.loc[quote.idxmax()].sum()})",
                  "p_asintotico": t["p_asintotico"], "p_permutazione": t["p_permutazione"], "V_cramer": t["V_cramer"]})
mancanti_segmenti = pd.DataFrame(righe)
mancanti_segmenti.to_csv(OUT_TAB / "step1_mancanti_per_segmento.csv", index=False)
with pd.option_context("display.max_colwidth", 80):
    display(mancanti_segmenti)

,skill,n_NA,tasso_SI_NA,n_valida,tasso_SI_valida,diff_punti,p_fisher,p_holm
12,Motivation,12,25.00,4367,57.04,-32.04,0.0375,0.4496
11,Responsibility,11,27.27,4367,57.04,-29.77,0.0651,0.7165
1,Teamwork,90,38.89,4289,57.33,-18.44,0.0005,0.0075
3,Critical Thinking,127,40.94,4252,57.43,-16.49,0.0003,0.0038
13,Feedback Management,12,41.67,4367,57.00,-15.33,0.3830,1.0000
2,Entrepreneurial Mindset,153,44.44,4225,57.40,-12.96,0.0020,0.0261
14,Interpersonal Skills,20,45.00,4358,57.00,-12.00,0.3658,1.0000
4,Creativity,96,47.92,4283,57.16,-9.24,0.0766,0.7659
10,Time Management,22,54.55,4357,56.97,-2.42,0.8321,1.0000
7,Cultural Awareness,49,55.10,4328,57.00,-1.90,0.8848,1.0000


,variabile,n_con_mancanti,media_con,mediana_con,n_senza,media_senza,mediana_senza,p_mann_whitney
0,OVERALL,470,9.00,9.0,3908,9.18,10.0,0.0002
1,DURATA (senza date Excel),470,87.76,77.0,3902,109.76,91.0,0.0000


,segmento,n,n_categorie,min,max,p_asintotico,p_permutazione,V_cramer
0,NAZIONE (Italia/estero),4377,2,ITALIA: 10.6% (362/3402),ESTERO: 11.1% (108/975),0.6982,0.7331,0.006
1,NAZIONE (opzione B),4377,11,SPAGNA: 3.0% (1/33),GRAN BRETAGNA E IRLANDA DEL NORD: 19.8% (16/81),0.1731,0.1719,0.057
2,AREA FUNZIONALE (Legale raggruppato),4334,12,"Produzione, logistica e qualità: 5.6% (4/72)",Ricerca & Sviluppo: 22.9% (89/388),0.0000,0.0005,0.140
3,CORSO,4285,24,B10: 4.3% (5/117),B09: 21.7% (36/166),0.0000,0.0005,0.131


In [21]:
# (g) Il legame mancanti–Hiring resta a parità di area, corso e OVERALL? (regressione logistica)
#     Campione: Hiring SI/NO, aree e corsi con n>=30, OVERALL valido. Linguaggio di associazione, non causale.
d = pd.DataFrame({"hiring": y.astype(int), "mancante": (k_miss > 0).astype(int),
                  "area": area_gr, "corso": co, "overall": ov})[camp & ov.notna()]
vc_a, vc_c = d.area.value_counts(), d.corso.value_counts()
d = d[d.area.isin(vc_a[vc_a >= SOGLIA_N].index) & d.corso.isin(vc_c[vc_c >= SOGLIA_N].index)]
righe = []
for nome, formula in [("non aggiustato", "hiring ~ mancante"),
                      ("aggiustato per area e corso", "hiring ~ mancante + C(area) + C(corso)"),
                      ("aggiustato per area, corso e OVERALL", "hiring ~ mancante + C(area) + C(corso) + overall")]:
    m = smf.logit(formula, d).fit(disp=0)
    lo, hi = np.exp(m.conf_int().loc["mancante"])
    righe.append({"modello": nome, "n": int(m.nobs), "OR_mancante": round(np.exp(m.params["mancante"]), 3),
                  "IC95_basso": round(lo, 3), "IC95_alto": round(hi, 3), "p": round(m.pvalues["mancante"], 4)})
logit_mancanti = pd.DataFrame(righe)
# Verifica indipendente: l'OR non aggiustato coincide con l'odds ratio della tabella 2x2 sullo stesso campione
t = pd.crosstab(d.mancante, d.hiring)
or_man = (t.loc[1, 1] * t.loc[0, 0]) / (t.loc[1, 0] * t.loc[0, 1])
assert np.isclose(or_man, logit_mancanti.OR_mancante.iloc[0], atol=5e-4)
logit_mancanti.to_csv(OUT_TAB / "step1_mancanti_vs_hiring_logit.csv", index=False)
display(logit_mancanti)
print(f"OR manuale (2x2) = {or_man:.3f}")

# (h) Quanto pesano i mancanti sulle medie? Media di ogni skill sui valori disponibili vs sulle sole righe complete
completi = n_sk == 16
sens = pd.DataFrame({
    "n_disponibili": m_valido[SKILLS].sum(), "media_disponibili": num[SKILLS].where(m_valido[SKILLS]).mean().round(3),
    "n_righe_complete": int(completi.sum()), "media_righe_complete": num[SKILLS][completi].mean().round(3)})
sens["diff"] = (sens.media_disponibili - sens.media_righe_complete).round(3)
sens = sens.loc[skill_con_na].sort_values("diff")
sens.to_csv(OUT_TAB / "step1_mancanti_sensibilita_medie.csv")
display(sens)

registra("Mancanti dichiarati (N/A), non completamente casuali: chi ha N/A ha meno Hiring e OVERALL più basso; quota diversa per area e corso",
         f"{len(skill_con_na)} skill su 16 (nessun N/A in Leadership e OVERALL)", r_na,
         "Mancante solo per quella skill (nessuna esclusione di riga, nessuna imputazione); n riportato per skill; confronto con le sole righe complete",
         f"Effetto sulle medie per skill piccolo (differenze tra {sens['diff'].min():+.3f} e {sens['diff'].max():+.3f}); "
         "le analisi su Hiring vanno lette sapendo che i mancanti non sono casuali", "limite da dichiarare", "da decidere")

,modello,n,OR_mancante,IC95_basso,IC95_alto,p
0,non aggiustato,4242,0.699,0.575,0.849,0.0003
1,aggiustato per area e corso,4242,0.729,0.595,0.893,0.0023
2,"aggiustato per area, corso e OVERALL",4242,0.798,0.647,0.984,0.0350


OR manuale (2x2) = 0.699


,n_disponibili,media_disponibili,n_righe_complete,media_righe_complete,diff
Creativity,4285,8.809,3905,8.832,-0.023
Teamwork,4291,9.149,3905,9.169,-0.020
Entrepreneurial Mindset,4227,8.752,3905,8.770,-0.018
Planning,4286,8.814,3905,8.829,-0.015
Flexibility,4369,8.957,3905,8.970,-0.013
Critical Thinking,4254,8.727,3905,8.740,-0.013
Feedback Management,4369,9.248,3905,9.261,-0.013
Writing,4375,9.057,3905,9.069,-0.012
Responsibility,4369,9.215,3905,9.225,-0.010
Digital,4316,9.210,3905,9.218,-0.008


### 1.7 Coerenza tra OVERALL e media delle skill

In [22]:
sk = num[SKILLS].where(m_valido[SKILLS])            # solo voti validi
media_sk = sk.mean(axis=1)                          # media delle skill valide
scarto = ov - media_sk                              # OVERALL − media skill
piatta16 = (n_sk == 16) & (sk.nunique(axis=1) == 1) # valutazione piatta (definita qui, usata anche in 1.8)

def coerenza(mask, nome):
    x, z = media_sk[mask], ov[mask]
    d = z - x
    fuori = (z < sk[mask].min(axis=1)) | (z > sk[mask].max(axis=1))
    arrot = np.floor(x + 0.5)                       # arrotondamento "commerciale" (0,5 verso l'alto)
    return {"campione": nome, "n": int(mask.sum()),
            "spearman_rho": round(stats.spearmanr(x, z).statistic, 3),
            "pearson_r": round(stats.pearsonr(x, z).statistic, 3),
            "scarto_medio": round(d.mean(), 3), "scarto_dev_std": round(d.std(), 3),
            "n_abs_scarto_ge_1": int((d.abs() >= 1).sum()), "pct_abs_scarto_ge_1": pct((d.abs() >= 1).sum(), len(d)),
            "n_abs_scarto_ge_2": int((d.abs() >= 2).sum()), "pct_abs_scarto_ge_2": pct((d.abs() >= 2).sum(), len(d)),
            "n_overall_fuori_min_max": int(fuori.sum()),
            "n_overall_uguale_media_arrotondata": int((z == arrot).sum()),
            "pct_overall_uguale_media_arrotondata": pct((z == arrot).sum(), len(d))}

base = ov.notna() & (n_sk >= 8)                     # media calcolata su almeno metà delle skill
compl = ov.notna() & (n_sk == 16)
coer = pd.DataFrame([coerenza(base, "A: OVERALL valido e ≥ 8 skill valide"),
                     coerenza(compl, "B: 16 skill valide"),
                     coerenza(compl & ~piatta16, "C: 16 skill valide, escluse le piatte")])
coer.to_csv(OUT_TAB / "step1_overall_vs_media_skill.csv", index=False)
display(coer.set_index("campione").T)

# Verifica indipendente: Pearson con la formula manuale
x, z = media_sk[base].to_numpy(), ov[base].to_numpy()
r_man = ((x - x.mean()) * (z - z.mean())).sum() / np.sqrt(((x - x.mean())**2).sum() * ((z - z.mean())**2).sum())
assert np.isclose(r_man, coer.pearson_r.iloc[0], atol=5e-4)
print(f"Pearson manuale (campione A) = {r_man:.4f}")

# Distribuzione dello scarto (campione A), classi con condizioni esplicite
sc = scarto[base]
condizioni = [sc <= -2, (sc > -2) & (sc <= -1), (sc > -1) & (sc < 1), (sc >= 1) & (sc < 2), sc >= 2]
etichette  = ["scarto ≤ −2", "−2 < scarto ≤ −1", "−1 < scarto < +1", "+1 ≤ scarto < +2", "scarto ≥ +2"]
distr_scarto = pd.DataFrame({"n": [int(c.sum()) for c in condizioni]}, index=etichette)
distr_scarto["pct"] = (100 * distr_scarto.n / len(sc)).round(2)
assert distr_scarto.n.sum() == len(sc)
assert distr_scarto.n.iloc[[0, 4]].sum() == coer.n_abs_scarto_ge_2.iloc[0]
assert distr_scarto.n.drop("−1 < scarto < +1").sum() == coer.n_abs_scarto_ge_1.iloc[0]
display(distr_scarto)

discordanti = df_raw.loc[base & (scarto.abs() >= 1), ["ID", "OVERALL", "Hiring"]].assign(
    media_skill=media_sk.round(2), n_skill_valide=n_sk, scarto=scarto.round(2)).sort_values("scarto")
discordanti.to_csv(OUT_TAB / "step1_overall_discordanti.csv", index=False)

registra("OVERALL distante dalla media delle skill: |scarto| ≥ 2", "OVERALL, 16 skill", base & (scarto.abs() >= 2),
         "Nessuna correzione; elencare i casi", "Possibili errori di compilazione non individuabili",
         "limite da dichiarare", "da decidere")
registra("OVERALL distante dalla media delle skill: 1 ≤ |scarto| < 2", "OVERALL, 16 skill",
         base & (scarto.abs() >= 1) & (scarto.abs() < 2),
         "Nessuna correzione: OVERALL è il giudizio complessivo del tutor (informazione del committente)",
         "Nessuno: è una discordanza moderata, compatibile con un giudizio autonomo", "limite da dichiarare", "da decidere")

campione,A: OVERALL valido e ≥ 8 skill valide,B: 16 skill valide,"C: 16 skill valide, escluse le piatte"
n,4377.000,3904.000,3050.000
spearman_rho,0.862,0.859,0.852
pearson_r,0.919,0.920,0.908
scarto_medio,0.099,0.106,0.136
scarto_dev_std,0.437,0.432,0.484
n_abs_scarto_ge_1,186.000,164.000,161.000
pct_abs_scarto_ge_1,4.250,4.200,5.280
n_abs_scarto_ge_2,8.000,7.000,7.000
pct_abs_scarto_ge_2,0.180,0.180,0.230
n_overall_fuori_min_max,9.000,7.000,4.000


Pearson manuale (campione A) = 0.9191


,n,pct
scarto ≤ −2,7,0.16
−2 < scarto ≤ −1,61,1.39
−1 < scarto < +1,4191,95.75
+1 ≤ scarto < +2,117,2.67
scarto ≥ +2,1,0.02


### 1.8 Valutazioni piatte (stesso voto su tutte le skill)

In [23]:
n_distinti = sk.nunique(axis=1)
piatta_parz = n_sk.between(8, 15) & (n_distinti == 1)     # identiche sulle skill valide, 8-15 valide
valore_piatto = sk.max(axis=1)

# Verifica indipendente (stessa logica di una formula Excel): 16 numeri e MIN = MAX sui valori grezzi
excel_like = (num[SKILLS].notna().sum(axis=1) == 16) & (num[SKILLS].min(axis=1) == num[SKILLS].max(axis=1))
assert (excel_like == piatta16).all()

n_compl = int((n_sk == 16).sum())
piatte = pd.DataFrame({
    "n": [int(piatta16.sum()), int(piatta_parz.sum())],
    "pct_sul_totale": [pct(piatta16.sum()), pct(piatta_parz.sum())],
    "den_righe_con_16_skill_valide": [n_compl, np.nan],
    "pct_su_righe_con_16_skill_valide": [pct(piatta16.sum(), n_compl), np.nan],
}, index=["piatta: 16 skill valide e identiche", "piatta parziale: 8-15 skill valide, tutte identiche"])
display(piatte)
print("Righe 'identiche sulle skill valide' per numero di skill valide (<16):",
      n_sk[(n_sk < 16) & (n_sk >= 1) & (n_distinti == 1)].value_counts().sort_index().to_dict())

per_valore = valore_piatto[piatta16].value_counts().sort_index(ascending=False).rename_axis("voto").to_frame("n")
per_valore["pct_delle_piatte"] = (100 * per_valore.n / piatta16.sum()).round(2)
per_valore["n_con_OVERALL_uguale"] = [int((piatta16 & (valore_piatto == v) & (ov == v)).sum()) for v in per_valore.index]
per_valore.to_csv(OUT_TAB / "step1_valutazioni_piatte.csv")
display(per_valore)
print("Piatte con OVERALL diverso dal voto piatto -> ID", ids(piatta16 & (ov != valore_piatto)))
print("Numero di voti distinti sulle 16 skill (righe con 16 skill valide):",
      n_distinti[n_sk == 16].value_counts().sort_index().to_dict())

# Peso delle piatte: su OVERALL medio e su Hiring (solo righe con 16 skill valide, per confrontare popolazioni omogenee)
peso = pd.DataFrame.from_dict({
    "piatte": {**tasso(piatta16), "OVERALL_medio": round(ov[piatta16].mean(), 3), "n_OVERALL": int(ov[piatta16].notna().sum())},
    "non piatte (16 skill valide)": {**tasso((n_sk == 16) & ~piatta16), "OVERALL_medio": round(ov[(n_sk == 16) & ~piatta16].mean(), 3),
                                     "n_OVERALL": int(ov[(n_sk == 16) & ~piatta16].notna().sum())}}, orient="index")
display(peso)
print(f"OVERALL medio su tutte le righe con OVERALL valido: {ov.mean():.3f} (n={int(ov.notna().sum())})")

registra("Valutazioni piatte (16 skill valide e identiche)", "16 skill", piatta16,
         "Tenere (non sono errori dimostrabili); flag e analisi di sensibilità con/senza",
         "Escluderle toglierebbe soprattutto giudizi 'tutto 10', abbasserebbe le medie e cambierebbe i tassi di Hiring; "
         "tenerle comprime le differenze tra skill", "limite da dichiarare", "da decidere")

,n,pct_sul_totale,den_righe_con_16_skill_valide,pct_su_righe_con_16_skill_valide
piatta: 16 skill valide e identiche,854,19.49,3905.0,21.87
"piatta parziale: 8-15 skill valide, tutte identiche",85,1.94,NaN,NaN


Righe 'identiche sulle skill valide' per numero di skill valide (<16): {1: 2, 9: 1, 10: 1, 12: 5, 13: 10, 14: 19, 15: 49}


,n,pct_delle_piatte,n_con_OVERALL_uguale
voto,,,
10.0,743,87.00,742
9.0,64,7.49,63
8.0,33,3.86,32
7.0,8,0.94,8
6.0,5,0.59,5
5.0,1,0.12,1


Piatte con OVERALL diverso dal voto piatto -> ID ['731', '1323', '1949']
Numero di voti distinti sulle 16 skill (righe con 16 skill valide): {1: 854, 2: 1194, 3: 1197, 4: 507, 5: 120, 6: 25, 7: 7, 8: 1}


,n,n_SI,tasso_SI_pct,IC95_basso,IC95_alto,OVERALL_medio,n_OVERALL
piatte,854,531,62.18,58.9,65.4,9.792,854
non piatte (16 skill valide),3049,1728,56.67,54.9,58.4,9.015,3050


OVERALL medio su tutte le righe con OVERALL valido: 9.163 (n=4380)


### 1.9 Tabella riassuntiva dell'audit

In [24]:
riepilogo = pd.DataFrame(audit)
riepilogo.to_csv(OUT_TAB / "step1_audit_riepilogo.csv", index=False)
with pd.option_context("display.max_colwidth", 120):
    display(riepilogo)

# Le righe della tabella si sovrappongono (una riga del dataset può avere più problemi):
# conteggio dell'unione delle righe con almeno un errore da correggere
errori = pd.concat([m for t, m in maschere_audit.values() if t == "errore da correggere"], axis=1).any(axis=1)
somma = sum(int(m.sum()) for t, m in maschere_audit.values() if t == "errore da correggere")
print(f"Righe con almeno un errore da correggere (unione, esclusi i cosmetici): {n_pct(errori.sum())} "
      f"[somma delle righe in tabella: {somma}] -> ID {ids(errori)}")
cosm = pd.concat([m for t, m in maschere_audit.values() if t == "errore cosmetico"], axis=1).any(axis=1)
print("Righe con solo errori cosmetici:", n_pct((cosm & ~errori).sum()))

# Controllo finale: nessun trattamento applicato (df_raw identico al file riletto da zero, ordine incluso)
pd.testing.assert_frame_equal(df_raw, rileggi_file())
print("df_raw invariato: nessun trattamento applicato.")

,problema,colonne,n_righe,pct_sul_totale,trattamento_proposto,rischio_del_trattamento,tipo,stato
0,Testo non numerico nei voti,"Cultural Awareness, Listening, Interpersonal Skills",3,0.07,Convertire in mancante solo per quella skill,Minimo: il voto reale è irrecuperabile,errore da correggere,già deciso
1,"Fuori scala ricodificabile (100→10, 999→9, 88→8, 55→5)","OVERALL, Writing, Cultural Awareness, Responsibility",4,0.09,Ricodificare con la regola 'cifra ripetuta → cifra; 100 → 10',Il valore reale potrebbe essere diverso (999 è spesso usato come codice di 'mancante'),errore da correggere,già deciso
2,Fuori scala non ricodificabile (7000),Entrepreneurial Mindset,1,0.02,Convertire in mancante solo per quella skill,Minimo: 1 cella,errore da correggere,già deciso
3,Riga quasi vuota,"tutte tranne ID, DURATA, SETTORE AZIENDALE",1,0.02,Escludere la riga,Nessuno rilevante: non contiene valutazioni,errore da correggere,già deciso
4,Valutazione con 1 sola skill valida (15 N/A),15 skill,2,0.05,Tenere per OVERALL e Hiring; escludere dai confronti tra skill e dalla coerenza OVERALL-skill,Nessuno rilevante (2 righe),limite da dichiarare,da decidere
5,ID duplicato,ID,0,0.00,Nessuno,-,nessun problema,già deciso
6,Righe identiche su tutte le colonne tranne ID (9 coppie),tutte tranne ID,18,0.41,Nessuno: ID diverso = stage diverso,"Se fossero doppi inserimenti, 9 valutazioni conterebbero due volte (attese per caso 6.2, osservate 9)",limite da dichiarare,già deciso
7,NAZIONE mancante (righe altrimenti complete),NAZIONE,2,0.05,Mancante solo per le analisi per paese,Minimo,errore da correggere,già deciso
8,Spazio finale nell'etichetta 'ONG ed Enti No Profit ',SETTORE AZIENDALE,131,2.99,Rimuovere gli spazi iniziali/finali,Nessuno (nessuna categoria si fonde),errore cosmetico,da decidere
9,"Convenzioni di nome non uniformi (CINA REPUBBLICA POPOLARE, CECA REPUBBLICA, IRLANDA - EIRE, SUDAFRICANA REPUBBLICA,...",NAZIONE,46,1.05,Nessuna modifica (cosmetico: nessun paese compare con due nomi),Nessuno,nessun problema,da decidere


Righe con almeno un errore da correggere (unione, esclusi i cosmetici): 20 (0.46% di 4382) [somma delle righe in tabella: 20] -> ID ['1', '42', '73', '98', '107', '475', '510', '816', '943', '1082', '1134', '1285', '1894', '2031', '3145', '3811', '3859', '3863', '3919', '4378']
Righe con solo errori cosmetici: 131 (2.99% di 4382)
df_raw invariato: nessun trattamento applicato.


### 1.10 Commento: errori da correggere e limiti da dichiarare

1. **Errori da correggere** (20 righe, 0,46%, già concordati): 3 testi non numerici, 5 fuori scala (4 ricodificabili, 1 mancante), la riga quasi vuota ID 816, 2 NAZIONE vuote, 2 Hiring vuoti o `Null`, 7 DURATA che sono date Excel del 2024. Più lo spazio finale in un'etichetta di SETTORE (cosmetico).
2. **Mancanti (`N/A`)**: 470 righe (10,7%). Non sono completamente casuali: chi ha `N/A` riceve meno offerte (49,4% su n=470 contro 57,9% su n=3.909) e la quota di `N/A` cambia molto per area e corso. Le medie per skill quasi non si muovono (differenze ≤ 0,023 punti), ma le letture su Hiring vanno fatte sapendo questo.
3. **Valutazioni piatte**: 854 righe (19,5%), quasi tutte "tutto 10" (743). Non sono errori dimostrabili, ma comprimono le differenze tra skill e hanno un tasso di Hiring più alto (62,2% su n=854 contro 56,7% su n=3.049): vanno tenute con flag e analisi di sensibilità.
4. **OVERALL** è molto coerente con la media delle skill (Spearman 0,86, n=4.377) ed è uguale alla media arrotondata nel 78% dei casi: aggiunge poca informazione oltre alle skill. Solo 8 casi hanno uno scarto ≥ 2.
5. **SETTORE AZIENDALE** è quasi indipendente da area, corso, paese e Hiring (V ≤ 0,10 contro 0,39 tra area e corso) ed è distribuito in modo quasi uniforme: limite da dichiarare, da non usare per segmentare.
6. **Altri limiti**: DURATA concentrata su pochi valori standard (179, 39, 88 giorni = 24% delle righe), 54 paesi e 13 corsi con n<30, etichette residuali in AREA (68 righe), MONACO ambiguo (4 righe).

### Registro assunzioni e limiti – aggiornato a fine Step 1

**Decisioni del committente (conversazione del 10/10/2026) – non ancora applicate, lo saranno nello step di pulizia**

| # | Decisione |
|---|---|
| D1 | `N/A` → mancante solo per quella skill; nessuna esclusione di riga, nessuna imputazione |
| D2 | Fuori scala: 100→10, 88→8, 55→5, 999→9; 7000 → mancante |
| D3 | Testi non numerici (`A`, `BBBB`, `NN`) → mancante |
| D4 | Escludere ID 816; celle vuote o `Null` in NAZIONE, AREA, CORSO, Hiring → mancante solo per quella variabile |
| D5 | DURATA ≥ 45.000 (date Excel) → mancante; 547 e 729 tenuti e segnalati |
| D6 | Righe identiche escluso ID: tenute (ID diverso = stage diverso) |
| D7 | CORSO "L" è un codice valido |
| D8 | Soglia n ≥ 30; NAZIONE: paesi con n ≥ 30 + "Altri esteri"; AREA: sottovoci "Legale - …" → "Legale"; CORSO: nessun accorpamento |
| D9 | OVERALL = giudizio complessivo del tutor; Hiring = SI significa offerta di assunzione dopo lo stage |
| D10 | Dataset fuori dal repository |

**Assunzioni**

| # | Assunzione | Base |
|---|---|---|
| A1 | Codifica cp1252, separatore `;` | verificata (Step 0) |
| A2 | NAZIONE = paese della sede dello stage | brief |
| A3 | CORSO = codice anonimo, nessuna ipotesi sul tipo di corso | brief |
| A4 | Voti interi 1-10 | brief; 99,99% delle celle numeriche |
| A5 | DURATA espressa in giorni | non scritta nel file; coerente con i valori (mediana 91, picco a 179) |
| A6 | Ogni ID è uno stage distinto | decisione D6 |

**Limiti**

| # | Limite | Evidenza (Step 1) |
|---|---|---|
| L1 | Mancanti non completamente casuali | Hiring 49,4% (n=470) vs 57,9% (n=3.909), p=0,0004; associazione che resta aggiustando per area, corso e OVERALL (OR 0,80, p=0,035, n=4.242); quota di N/A per area da 5,6% a 22,9% |
| L2 | SETTORE AZIENDALE quasi indipendente dalle altre variabili | V di Cramér ≤ 0,10 con area, corso, paese, Hiring; controllo positivo AREA × CORSO V=0,39 |
| L3 | Valutazioni piatte ed effetto tetto | 854 piatte (19,5%), 743 "tutto 10"; OVERALL medio 9,16 (n=4.380) |
| L4 | OVERALL poco informativo oltre le skill | uguale alla media arrotondata nel 77,8% dei casi (n=4.377) |
| L5 | DURATA con valori standard | 179 gg in 556 righe (12,7%); 39 gg in 266; 88 gg in 226 |
| L6 | Segmenti piccoli | 54 paesi (294 righe), 13 corsi (95 righe), 3 aree (45 righe) con n < 30 |
| L7 | Etichette ambigue o residuali | MONACO (4 righe); Generico, Altro, Legale - Ulteriori aree (68 righe) |
| L8 | Possibili coincidenze tra righe | 9 coppie identiche escluso ID (attese per caso 6,2; P=0,17); 3 coppie identiche tranne Hiring |
| L9 | ID non contigui (6 mancanti) | Step 0 |
| L10 | Dataset sintetico | brief |

**Ancora da decidere:** spazio finale in SETTORE; DURATA 366-400 e < 30 senza flag; valutazioni con una sola skill valida (ID 2482, 3563); uso di SETTORE; MONACO; trattamento delle valutazioni piatte; come dichiarare i mancanti non casuali; discordanze OVERALL-skill.